Since there is a retraining needed for poor accuracy values. So to hold up the FGP masks we need a interactive environment to retain and experiment with diffrent parameters.

In [1]:

import sys
import torch
from torch import nn
import copy
import random
import torch
from pathlib import Path

# Notebook is inside .../PruningNAS/PruningNAS, so add project root (.../PruningNAS)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Install local package in editable mode (run once; safe to re-run)
%pip install -e ..


from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Utills.EvaluatiorUtills import get_model_size, get_sparsity
from PruningNAS.Utills.PrunUtillCP import ChannelPrunner
from PruningNAS.Utills.PrunUtillFGP import FineGrainedPruner
from PruningNAS.Utills.TrainingModulesUtills import TrainingPrunned, evaluate
from PruningNAS.Utills.Utill import print_model
from PruningNAS.Utills.ViewerUtills import plot_accuracy, plot_loss  # Ensure you import your correct model architecture
seed=0
random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

Obtaining file:///D:/Sutanu_WorkSpace/PruningNas
Note: you may need to restart the kernel to use updated packages.
Device: cuda


ERROR: file:///D:/Sutanu_WorkSpace/PruningNas does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set Static Params here:

In [3]:
import os


current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")


Current working directory: d:\Sutanu_WorkSpace\PruningNAS\PruningNAS


In [ ]:

# Initialize the model
basedir=''
path='./dataset/cifar10'
select_model='MobilenetV1'
pruning_type='FGP'
#model_path='./checkpoint/vgg_mrl_99.51375579833984.pth'
model_path=r'.\checkpoint\MobilenetV1\MobilenetV1_cifar_94.129997.pth'
# Load the saved state_dict correctly
model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)

sparsity_dict   = {

"model.0": 0.7,

"model.3.depthwise": 0.6,
"model.3.pointwise": 0.7,

"model.4.depthwise": 0.4,
"model.4.pointwise": 0.6,

"model.5.depthwise": 0.3,
"model.5.pointwise": 0.6,

"model.6.depthwise": 0.4,
"model.6.pointwise": 0.6,

"model.7.depthwise": 0.5,
"model.7.pointwise": 0.6,

"model.8.depthwise": 0.4,
"model.8.pointwise": 0.7,

"model.9.depthwise": 0.6,
"model.9.pointwise": 0.6,

"model.10.depthwise": 0.6,
"model.10.pointwise": 0.6,

"model.11.depthwise": 0.6,
"model.11.pointwise": 0.7,

"model.12.depthwise": 0.6,
"model.12.pointwise": 0.8,

"model.13.depthwise": 0.7,
"model.13.pointwise": 0.9,

"model.14.depthwise": 0.8,
"model.14.pointwise": 0.9,

"model.15.depthwise": 0.9,
"model.15.pointwise": 0.9,

"fc": 0.9
}
# sparsity_dict ={ 
# 'conv1':0.80,
# 'layer1.0.conv1':0.90,
# 'layer1.0.conv2':0.90,
# 'layer1.1.conv1':0.90,
# 'layer1.1.conv2':0.90,
# 'layer1.2.conv1':0.90,
# 'layer1.2.conv2':0.90,
# 'layer2.0.conv1':0.90,
# 'layer2.0.conv2':0.80,
# 'layer2.0.shortcut.0':0.80,
# 'layer2.1.conv1':0.90,
# 'layer2.1.conv2':0.90,
# 'layer2.2.conv1':0.90,
# 'layer2.2.conv2':0.90,
# 'layer2.3.conv1':0.90,
# 'layer2.3.conv2':0.90,
# 'layer3.0.conv1':0.90,
# 'layer3.0.conv2':0.80,
# 'layer3.0.shortcut.0':0.80,
# 'layer3.1.conv1':0.90,
# 'layer3.1.conv2':0.90,
# 'layer3.2.conv1':0.90,
# 'layer3.2.conv2':0.90,
# 'layer3.3.conv1':0.90,
# 'layer3.3.conv2':0.90,
# 'layer3.4.conv1':0.90,
# 'layer3.4.conv2':0.90,
# 'layer3.5.conv1':0.90,
# 'layer3.5.conv2':0.90,
# 'layer4.0.conv1':0.90,
# 'layer4.0.conv2':0.90,
# 'layer4.0.shortcut.0':0.90,
# 'layer4.1.conv1':0.90,
# 'layer4.1.conv2':0.90,
# 'layer4.2.conv1':0.90,
# 'layer4.2.conv2':0.90,
# 'fc':0.90,}


Define experimental params here:

In [5]:
num_finetune_epochs = 400
lr=0.1

In [6]:
train_dataloader,test_dataloader=get_dataloaders(path, batch_size=256 ) # Basemodel
dense_model_accuracy=evaluate(model,test_dataloader)
print('dense_model_accuracy:',dense_model_accuracy)


d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
                                                     

dense_model_accuracy: (95.7699966430664, 0.0)


In [7]:
pruned_model=copy.deepcopy(model)
if pruning_type=='FGP':
    isCallback=True
    pruner = FineGrainedPruner(pruned_model, sparsity_dict)
elif pruning_type=='CP':
    pruned_model=ChannelPrunner(pruned_model, sparsity_dict,select_model)
    pruner=None
    isCallback=False
else:
    print('pruning_type doesn\'t exists')
    exit

print_model(pruned_model)
print(f'The sparsity of each layer becomes')
for name, param in pruned_model.named_parameters():
    print(f'  {name}: {get_sparsity(param):.2f}')


init_conv.0.weight torch.Size([64, 3, 3, 3])
init_conv.1.weight torch.Size([64])
init_conv.1.bias torch.Size([64])
blocks.0.block.0.norm1.weight torch.Size([64])
blocks.0.block.0.norm1.bias torch.Size([64])
blocks.0.block.0.conv1.weight torch.Size([77, 64, 1, 1])
blocks.0.block.0.norm2.weight torch.Size([77])
blocks.0.block.0.norm2.bias torch.Size([77])
blocks.0.block.0.conv2.weight torch.Size([32, 77, 3, 3])
blocks.0.block.1.norm1.weight torch.Size([96])
blocks.0.block.1.norm1.bias torch.Size([96])
blocks.0.block.1.conv1.weight torch.Size([90, 96, 1, 1])
blocks.0.block.1.norm2.weight torch.Size([90])
blocks.0.block.1.norm2.bias torch.Size([90])
blocks.0.block.1.conv2.weight torch.Size([32, 90, 3, 3])
blocks.0.block.2.norm1.weight torch.Size([128])
blocks.0.block.2.norm1.bias torch.Size([128])
blocks.0.block.2.conv1.weight torch.Size([51, 128, 1, 1])
blocks.0.block.2.norm2.weight torch.Size([51])
blocks.0.block.2.norm2.bias torch.Size([51])
blocks.0.block.2.conv2.weight torch.Size([32,

In [8]:
model_path=r'.\checkpoint\Densenet-121\CP\Densenet-121_cifar_CP_94.58999633789062.pth'
# Load the saved state_dict correctly
pruned_model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

pruned_model.to(device)

DenseNet(
  (init_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (blocks): ModuleList(
    (0): DenseBlock(
      (block): Sequential(
        (0): DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (1): DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(96, 12

In [9]:

dense_model_size = get_model_size(model, count_nonzero_only=True)
sparse_model_size = get_model_size(pruned_model, count_nonzero_only=True)

print(f"Sparse model has size={sparse_model_size:.2f} MiB = {sparse_model_size / dense_model_size * 100:.2f}% of dense model size")
sparse_model_accuracy,_ = evaluate(pruned_model, test_dataloader)
print(f"Sparse model has accuracy={sparse_model_accuracy:.2f}% before fintuning")
lr
optimizer = torch.optim.SGD(pruned_model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_finetune_epochs)
criterion = nn.CrossEntropyLoss()

Sparse model has size=12.33 MiB = 46.47% of dense model size


Sparse model has accuracy=94.59% before fintuning


In [ ]:
pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)
print(pruned_model_accuracy)


In [12]:
print(pruned_model_accuracy)
basedir='.'

torch.save(best_pruned_model, f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}_{pruned_model_accuracy}.pth')

titel_append=f'of {pruning_type} based Pruned {select_model.title()} model'
save_path=f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}'

plot_accuracy(accuracies,titel_append=titel_append,save_path=save_path+'_acc.png' )
plot_loss(train_losses,test_losses,titel_append=titel_append,save_path=save_path+'_loss.png')

94.75999450683594
[94.47000122070312, 94.3699951171875, 94.43999481201172, 94.40999603271484, 94.4000015258789, 94.41999816894531, 94.38999938964844, 94.54000091552734, 94.47999572753906, 94.41999816894531, 94.47999572753906, 94.50999450683594, 94.47999572753906, 94.57999420166016, 94.41999816894531, 94.40999603271484, 94.47999572753906, 94.38999938964844, 94.52999877929688, 94.45999908447266, 94.48999786376953, 94.44999694824219, 94.4000015258789, 94.54999542236328, 94.41999816894531, 94.56999969482422, 94.3699951171875, 94.41999816894531, 94.5199966430664, 94.5999984741211, 94.38999938964844, 94.52999877929688, 94.5, 94.5999984741211, 94.61000061035156, 94.52999877929688, 94.45999908447266, 94.43999481201172, 94.47000122070312, 94.52999877929688, 94.48999786376953, 94.52999877929688, 94.5999984741211, 94.5199966430664, 94.43999481201172, 94.47000122070312, 94.41999816894531, 94.47000122070312, 94.54000091552734, 94.48999786376953, 94.4000015258789, 94.43000030517578, 94.4700012207031

In [10]:
best_pruned_model=pruned_model
best_pruned_model.cuda()

DenseNet(
  (init_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (blocks): ModuleList(
    (0): DenseBlock(
      (block): Sequential(
        (0): DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (1): DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(96, 12

In [11]:

num_finetune_epochs=300
optimizer = torch.optim.SGD(best_pruned_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(best_pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)


    Epoch 252 Test accuracy:94.64% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2442


    Epoch 253 Test accuracy:94.59% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2432


    Epoch 254 Test accuracy:94.58% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2451


    Epoch 255 Test accuracy:94.55% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2434


    Epoch 256 Test accuracy:94.46% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2452


...

    Epoch 296 Test accuracy:94.55% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2429


    Epoch 297 Test accuracy:94.62% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2435


    Epoch 298 Test accuracy:94.59% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2439


    Epoch 299 Test accuracy:94.61% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2434


    Epoch 300 Test accuracy:94.56% / Best Accuracy: 94.76%, train loss: 0.0004, test loss: 0.2424
